# Mini-Projet de Data Science

Ce notebook répond à un exercice de certification en Data Science, comprenant deux étapes principales :
- Une analyse et prédiction des données de test à partir d'un modèle de régression.
- Une classification des données d'apprentissage en plusieurs groupes, suivie d'une modélisation spécifique pour chaque groupe.

Les données d'apprentissage sont stockées dans `donapp.csv` et les données de test dans `dontest.csv`.

In [ ]:
import pandas as pd

# Charger les jeux de données
donapp = pd.read_csv('/mnt/data/donapp.csv')
dontest = pd.read_csv('/mnt/data/dontest.csv')

# Afficher les premières lignes pour comprendre la structure des données
donapp.head(), dontest.head()

## Analyse exploratoire des données

Nous commençons par explorer les données pour mieux comprendre les caractéristiques.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualisation des distributions des variables pour explorer les données
donapp.describe()

# Visualiser la distribution de la variable cible 'Y' pour l'analyse initiale
plt.figure(figsize=(10, 6))
sns.histplot(donapp['Y'], kde=True)
plt.title("Distribution de la variable cible 'Y'")
plt.xlabel("Y")
plt.ylabel("Densité")
plt.show()

# Visualisation de la corrélation entre les variables pour identifier les plus pertinentes
plt.figure(figsize=(15, 12))
sns.heatmap(donapp.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Carte de corrélation entre les variables")
plt.show()

## Modélisation et prédiction initiale

Nous choisissons un modèle de régression linéaire de base pour effectuer une première estimation de la MSE sur les données de test.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Séparer les caractéristiques (X) et la cible (Y) pour le jeu de données d'apprentissage
X_train = donapp.drop(columns=['Y'])
y_train = donapp['Y']

# Pour les données de test
X_test = dontest.drop(columns=['Y'])
y_test = dontest['Y']

# Modèle de régression linéaire de base
model = LinearRegression()
model.fit(X_train, y_train)

# Prédictions et calcul de la MSE
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mse

## Classification en groupes et modélisation par groupe

Nous effectuons une classification en 3 groupes avec K-means, puis ajustons un modèle de régression pour chaque groupe afin d'améliorer les prédictions.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Standardisation des données pour une meilleure performance de K-means
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Classification en 3 groupes avec K-means
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_train_scaled)

# Ajout des clusters aux données d'apprentissage
donapp['Cluster'] = clusters

# Visualisation de la distribution des clusters par rapport à la variable cible 'Y'
plt.figure(figsize=(10, 6))
sns.boxplot(x=donapp['Cluster'], y=donapp['Y'])
plt.title("Distribution de 'Y' par cluster")
plt.xlabel("Cluster")
plt.ylabel("Y")
plt.show()

# Vérifier la taille des clusters
donapp['Cluster'].value_counts()

In [ ]:
from sklearn.metrics import mean_squared_error

# Préparer un dictionnaire pour stocker les modèles et erreurs pour chaque cluster
models = {}
mse_per_cluster = {}

# Appliquer le même clustering au jeu de données de test
X_test_scaled = scaler.transform(X_test)
test_clusters = kmeans.predict(X_test_scaled)

# Pour chaque cluster, entraîner un modèle et faire les prédictions
for cluster_id in donapp['Cluster'].unique():
    # Extraire les données du cluster
    cluster_data = donapp[donapp['Cluster'] == cluster_id]
    X_cluster = cluster_data.drop(columns=['Y', 'Cluster'])
    y_cluster = cluster_data['Y']

    # Entraîner un modèle de régression pour ce cluster
    model = LinearRegression()
    model.fit(X_cluster, y_cluster)

    # Stocker le modèle
    models[cluster_id] = model

    # Filtrer les données de test pour le cluster actuel
    X_test_cluster = X_test[test_clusters == cluster_id]
    y_test_cluster = y_test[test_clusters == cluster_id]

    # Faire les prédictions et calculer l'erreur pour le cluster
    y_pred_cluster = model.predict(X_test_cluster)
    mse_cluster = mean_squared_error(y_test_cluster, y_pred_cluster)

    # Stocker la MSE pour le cluster
    mse_per_cluster[cluster_id] = mse_cluster

mse_per_cluster